## DINOv2 LSTM ##

In [1]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm

from PIL import Image
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix


import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import Dataset, DataLoader


import os
import pickle
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from PIL import Image

/media/osero/SamsungSSD/miniconda_files/conda/envs/dinov2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Device

In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# device = torch.device("cpu")
device

device(type='cuda')

# Test Models

In [ ]:

# from torch.nn import Linear, Conv2d, BatchNorm1d, BatchNorm2d, PReLU, Sequential, Module
# import torch
# import torch.nn as nn
# class Flatten(Module):
#     def forward(self, input):
#         return input.view(input.size(0), -1)

# def l2_norm(input,axis=1):
#     norm = torch.norm(input,2,axis,True)
#     output = torch.div(input, norm)
#     return output

# class Conv_block(Module):
#     def __init__(self, in_c, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0), groups=1):
#         super(Conv_block, self).__init__()
#         self.conv = Conv2d(in_c, out_channels=out_c, kernel_size=kernel, groups=groups, stride=stride, padding=padding, bias=False)
#         self.bn = BatchNorm2d(out_c)
#         self.prelu = PReLU(out_c)
#     def forward(self, x):
#         x = self.conv(x)
#         x = self.bn(x)
#         x = self.prelu(x)
#         return x

# class Linear_block(Module):
#     def __init__(self, in_c, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0), groups=1):
#         super(Linear_block, self).__init__()
#         self.conv = Conv2d(in_c, out_channels=out_c, kernel_size=kernel, groups=groups, stride=stride, padding=padding, bias=False)
#         self.bn = BatchNorm2d(out_c)
#     def forward(self, x):
#         x = self.conv(x)
#         x = self.bn(x)
#         return x

# class Depth_Wise(Module):
#      def __init__(self, in_c, out_c, residual = False, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=1):
#         super(Depth_Wise, self).__init__()
#         self.conv = Conv_block(in_c, out_c=groups, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
#         self.conv_dw = Conv_block(groups, groups, groups=groups, kernel=kernel, padding=padding, stride=stride)
#         self.project = Linear_block(groups, out_c, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
#         self.residual = residual
#      def forward(self, x):
#         if self.residual:
#             short_cut = x
#         x = self.conv(x)
#         x = self.conv_dw(x)
#         x = self.project(x)
#         if self.residual:
#             output = short_cut + x
#         else:
#             output = x
#         return output
  

# class Swish(nn.Module):
#     def __init__(self):
#         super(Swish, self).__init__()

#         self.sigmoid = nn.Sigmoid()

#     def forward(self, x):
#         return x * self.sigmoid(x)

# NON_LINEARITY = {
#     'ReLU': nn.ReLU(inplace=True),
#     'Swish': Swish(),
# }        


# class h_sigmoid(nn.Module):
#     def __init__(self, inplace=True):
#         super(h_sigmoid, self).__init__()
#         self.relu = nn.ReLU6(inplace=inplace)

#     def forward(self, x):
#         return self.relu(x + 3) / 6

# class h_swish(nn.Module):
#     def __init__(self, inplace=True):
#         super(h_swish, self).__init__()
#         self.sigmoid = h_sigmoid(inplace=inplace)

#     def forward(self, x):
#         return x * self.sigmoid(x)

# class swish(nn.Module):
#     def forward(self, x):
#         return x * torch.sigmoid(x)
    
# class CoordAtt(nn.Module):
#     def __init__(self, inp, oup, groups=32):
#         super(CoordAtt, self).__init__()
#         self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
#         self.pool_w = nn.AdaptiveAvgPool2d((1, None))

#         mip = max(8, inp // groups)

#         self.conv1 = nn.Conv2d(inp, mip, kernel_size=1, stride=1, padding=0)
#         self.bn1 = nn.BatchNorm2d(mip)
#         self.conv2 = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0)
#         self.conv3 = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0)
#         self.relu = h_swish()

#     def forward(self, x):
#         identity = x
#         n,c,h,w = x.size()
#         x_h = self.pool_h(x)
#         x_w = self.pool_w(x).permute(0, 1, 3, 2)

#         y = torch.cat([x_h, x_w], dim=2)
#         y = self.conv1(y)
#         y = self.bn1(y)
#         y = self.relu(y) 
#         x_h, x_w = torch.split(y, [h, w], dim=2)
#         x_w = x_w.permute(0, 1, 3, 2)

#         x_h = self.conv2(x_h).sigmoid()
#         x_w = self.conv3(x_w).sigmoid()
#         x_h = x_h.expand(-1, -1, h, w)
#         x_w = x_w.expand(-1, -1, h, w)

#         y = identity * x_w * x_h

#         return y

        
# class MDConv(Module):
#     def __init__(self, channels, kernel_size, split_out_channels, stride):
#         super(MDConv, self).__init__()
#         self.num_groups = len(kernel_size)
#         self.split_channels = split_out_channels
#         self.mixed_depthwise_conv = nn.ModuleList()
#         for i in range(self.num_groups):
#             self.mixed_depthwise_conv.append(Conv2d(
#                 self.split_channels[i],
#                 self.split_channels[i],
#                 kernel_size[i],
#                 stride=stride,
#                 padding=kernel_size[i]//2,
#                 groups=self.split_channels[i],
#                 bias=False
#             ))
#         self.bn = BatchNorm2d(channels)
#         self.prelu = PReLU(channels)            
       
#     def forward(self, x):
#         if self.num_groups == 1:
#             return self.mixed_depthwise_conv[0](x)

#         x_split = torch.split(x, self.split_channels, dim=1)
#         x = [conv(t) for conv, t in zip(self.mixed_depthwise_conv, x_split)]
#         x = torch.cat(x, dim=1)

#         return x        
      
        
# class Mix_Depth_Wise(Module):
#      def __init__(self, in_c, out_c, residual = False, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=1, kernel_size=[3,5,7], split_out_channels=[64,32,32]):
#         super(Mix_Depth_Wise, self).__init__()
#         self.conv = Conv_block(in_c, out_c=groups, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
#         self.conv_dw = MDConv(channels=groups, kernel_size=kernel_size, split_out_channels=split_out_channels, stride=stride)
#         self.CA = CoordAtt(groups, groups)
#         self.project = Linear_block(groups, out_c, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
#         self.residual = residual
#      def forward(self, x):
#         if self.residual:
#             short_cut = x
#         x = self.conv(x)
#         x = self.conv_dw(x)
#         x = self.CA(x)
#         x = self.project(x)
#         if self.residual:
#             output = short_cut + x
#         else:
#             output = x
#         return output
          
# class Residual(Module):
#     def __init__(self, c, num_block, groups, kernel=(3, 3), stride=(1, 1), padding=(1, 1)):
#         super(Residual, self).__init__()
#         modules = []
#         for _ in range(num_block):
#             modules.append(Depth_Wise(c, c, residual=True, kernel=kernel, padding=padding, stride=stride, groups=groups))
#         self.model = Sequential(*modules)
#     def forward(self, x):
#         return self.model(x)
        
# class Mix_Residual(Module):
#     def __init__(self, c, num_block, groups, kernel=(3, 3), stride=(1, 1), padding=(1, 1), kernel_size=[3,5], split_out_channels=[64,64]):
#         super(Mix_Residual, self).__init__()
#         modules = []
#         for _ in range(num_block):
#             modules.append(Mix_Depth_Wise(c, c, residual=True, kernel=kernel, padding=padding, stride=stride, groups=groups, kernel_size=kernel_size, split_out_channels=split_out_channels ))
#         self.model = Sequential(*modules)
#     def forward(self, x):
#         return self.model(x)
        

# class MixedFeatureNet(Module):
#     def __init__(self, embedding_size=256, out_h=7, out_w=7):
#         super(MixedFeatureNet, self).__init__()
#         #112x112
#         self.conv1 = Conv_block(3, 64, kernel=(3, 3), stride=(2, 2), padding=(1, 1))
#         #56x56
#         self.conv2_dw = Conv_block(64, 64, kernel=(3, 3), stride=(1, 1), padding=(1, 1), groups=64)
#         self.conv_23 = Mix_Depth_Wise(64, 64, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=128, kernel_size=[3,5,7], split_out_channels=[64,32,32] )
        
#         #28x28
#         self.conv_3 = Mix_Residual(64, num_block=9, groups=128, kernel=(3, 3), stride=(1, 1), padding=(1, 1), kernel_size=[3,5], split_out_channels=[96,32])
#         self.conv_34 = Mix_Depth_Wise(64, 128, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=256, kernel_size=[3,5,7],split_out_channels=[128,64,64] )
        
#         #14x14
#         self.conv_4 = Mix_Residual(128, num_block=16, groups=256, kernel=(3, 3), stride=(1, 1), padding=(1, 1), kernel_size=[3,5], split_out_channels=[192,64])
#         self.conv_45 = Mix_Depth_Wise(128, 256, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=512*2, kernel_size=[3,5,7,9],split_out_channels=[128*2,128*2,128*2,128*2] )
#         #7x7
#         self.conv_5 = Mix_Residual(256, num_block=6, groups=512, kernel=(3, 3), stride=(1, 1), padding=(1, 1), kernel_size=[3,5,7], split_out_channels=[86*2,85*2,85*2])                
#         self.conv_6_sep = Conv_block(256, 512, kernel=(1, 1), stride=(1, 1), padding=(0, 0))
#         self.conv_6_dw = Linear_block(512, 512, groups=512, kernel=(out_h, out_w), stride=(1, 1), padding=(0, 0))
#         self.conv_6_flatten = Flatten()
#         self.linear = Linear(512, embedding_size, bias=False)
#         self.bn = BatchNorm1d(embedding_size)
    
#     def forward(self, x):
#         out = self.conv1(x)
#         out = self.conv2_dw(out)
#         out = self.conv_23(out)
#         out = self.conv_3(out)
#         out = self.conv_34(out)
#         out = self.conv_4(out)
#         out = self.conv_45(out)
#         out = self.conv_5(out)
#         out = self.conv_6_sep(out)
#         out = self.conv_6_dw(out)
#         out = self.conv_6_flatten(out)
#         out = self.linear(out)
#         out = self.bn(out)

#         return l2_norm(out)


In [ ]:
# from torch import nn
# import torch
# from torch.nn import Module
# import os
# class Linear_block(Module):
#     def __init__(self, in_c, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0), groups=1):
#         super(Linear_block, self).__init__()
#         self.conv = nn.Conv2d(in_c, out_channels=out_c, kernel_size=kernel, groups=groups, stride=stride, padding=padding, bias=False)
#         self.bn = nn.BatchNorm2d(out_c)
#     def forward(self, x):
#         x = self.conv(x)
#         x = self.bn(x)
#         return x

# class Flatten(Module):
#     def forward(self, input):
#         return input.view(input.size(0), -1)
        
# class DDAMNet(nn.Module):
#     def __init__(self, num_class=7,num_head=2, pretrained=True):
#         super(DDAMNet, self).__init__()

#         net = MixedFeatureNet()
                
#         if pretrained:
#             net = torch.load("/home/osero/Desktop/CMPE/dinov2/classsification/denemeler/DDAMFN/MFN_msceleb.pth")
      
#         self.features = nn.Sequential(*list(net.children())[:-4])
#         self.num_head = num_head
#         for i in range(int(num_head)):
#             setattr(self,"cat_head%d" %(i), CoordAttHead())                  
      
#         self.Linear = Linear_block(512, 512, groups=512, kernel=(7, 7), stride=(1, 1), padding=(0, 0))
#         self.flatten = Flatten()      
#         self.fc = nn.Linear(512, num_class)
#         self.bn = nn.BatchNorm1d(num_class)
        
#     def forward(self, x):
#         x = self.features(x)
#         heads = []
       
#         for i in range(self.num_head):
#             heads.append(getattr(self,"cat_head%d" %i)(x))
#         head_out =heads
        
#         y = heads[0]
        
#         for i in range(1,self.num_head):
#             y = torch.max(y,heads[i])                     
        
#         y = x*y
#         y = self.Linear(y)
#         y = self.flatten(y) 
#         out = self.fc(y)        
#         return out, x, head_out
        
# class h_sigmoid(nn.Module):
#     def __init__(self, inplace=True):
#         super(h_sigmoid, self).__init__()
#         self.relu = nn.ReLU6(inplace=inplace)
#     def forward(self, x):
#         return self.relu(x + 3) / 6
                      
# class h_swish(nn.Module):
#     def __init__(self, inplace=True):
#         super(h_swish, self).__init__()
#         self.sigmoid = h_sigmoid(inplace=inplace)
#     def forward(self, x):
#         return x * self.sigmoid(x)

# class CoordAttHead(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.CoordAtt = CoordAtt(512,512)
#     def forward(self, x):
#         ca = self.CoordAtt(x)
#         return ca  
        
# class CoordAtt(nn.Module):
#     def __init__(self, inp, oup, groups=32):
#         super(CoordAtt, self).__init__()
      
#         self.Linear_h = Linear_block(inp, inp, groups=inp, kernel=(1, 7), stride=(1, 1), padding=(0, 0))        
#         self.Linear_w = Linear_block(inp, inp, groups=inp, kernel=(7, 1), stride=(1, 1), padding=(0, 0))
        
#         mip = max(8, inp // groups)

#         self.conv1 = nn.Conv2d(inp, mip, kernel_size=1, stride=1, padding=0)
#         self.bn1 = nn.BatchNorm2d(mip)
#         self.conv2 = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0)
#         self.conv3 = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0)
#         self.relu = h_swish()
#         self.Linear = Linear_block(oup, oup, groups=oup, kernel=(7, 7), stride=(1, 1), padding=(0, 0))
#         self.flatten = Flatten() 

#     def forward(self, x):
#         identity = x
#         n,c,h,w = x.size()
#         x_h = self.Linear_h(x)
#         x_w = self.Linear_w(x)
#         x_w = x_w.permute(0, 1, 3, 2)

#         y = torch.cat([x_h, x_w], dim=2)
#         y = self.conv1(y)
#         y = self.bn1(y)
#         y = self.relu(y) 
#         x_h, x_w = torch.split(y, [h, w], dim=2)
#         x_w = x_w.permute(0, 1, 3, 2)

#         x_h = self.conv2(x_h).sigmoid()
#         x_w = self.conv3(x_w).sigmoid()
#         x_h = x_h.expand(-1, -1, h, w)
#         x_w = x_w.expand(-1, -1, h, w)
        
#         y = x_w * x_h
 
#         return y


## Load DinoV2

In [3]:
from networks.DDAM import DDAMNet


model = DDAMNet(num_class=7, num_head=2, pretrained=True)

# checkpoint = torch.load(args.model_path, map_location=device)
# model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()        

aa = 4

In [11]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
data_transforms_val = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])])    
image1 = Image.open('/media/osero/SamsungSSD/CMPE_SSD/frame-face-c256/0001/User_2_001/001.jpg')
image2 = Image.open('/media/osero/SamsungSSD/CMPE_SSD/frame-face-c256/0001/User_2_001/002.jpg')
image1 = data_transforms_val(image1).to(device)
image2 = data_transforms_val(image2).to(device)
torch_test = torch.stack([image1, image2])

result = model(torch_test)
aaa = 4

KeyboardInterrupt: 

## Prepare Dataset

In [ ]:
# pose_pickle_folder = '/media/osero/SamsungSSD/CMPE_SSD/mmpose-full/0001/User_2_001.pickle'
pose_pickle_folder = '/media/osero/SamsungSSD/CMPE_SSD/mmpose-full/'

def get_active_frames_from_pickle(input_raw) -> np.ndarray:
    threshold = (
        (((input_raw["pose"]["left_hip"][:, 1] + input_raw["pose"]["right_hip"][:, 1]) / 2 )* 7)
        + input_raw["pose"]["nose"][:, 1] * 3
    ) / 10

    active_frames = (
        np.minimum(
            input_raw["hand_left"]["left_lunate_bone"][:, 1],
            input_raw["hand_right"]["right_lunate_bone"][:, 1],
        )
        < threshold
    )

    active_frame_indices = np.argwhere(active_frames).squeeze()
    return active_frame_indices


def get_active_frames(label_name, sample_name):
    pickle_file_name = f"{pose_pickle_folder}/{label_name}/{sample_name}.pickle"
    file = open(pickle_file_name, 'rb')
    input_raw = pickle.load(file)

    return get_active_frames_from_pickle(input_raw)

In [ ]:
####### SECOND #######


frame_frequency = 1

def create_label_dict(classes):
    label_dict = {}
    for i in range(0,len(classes)):
        label_dict[classes[i]] = i
    return label_dict

class CustomImageDataset(Dataset):
    def __init__(self, left_root_dir, right_root_dir):
        
        left_pickle_file = open(left_root_dir, 'rb')
        left_paths, left_features,left_labels = pickle.load(left_pickle_file)

        right_pickle_file = open(right_root_dir, 'rb')
        right_paths, right_features,right_labels = pickle.load(right_pickle_file)

        self.left_features = left_features
        self.right_features = right_features
        self.paths = left_paths
        self.classes = np.unique(left_labels)
        label_dict = create_label_dict(self.classes)
        self.labels = [label_dict[x] for x in left_labels]

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        splited_paths = self.paths[idx].split('/')

        active_frame_indices = get_active_frames(splited_paths[-2],splited_paths[-1])
        active_frame_indices = (
            active_frame_indices
            if active_frame_indices.size > 10
            else np.arange(0, len(self.left_features[idx]))
        )
        left_embeddings = [self.left_features[idx][i] for i in active_frame_indices]
        right_embeddings = [self.right_features[idx][i] for i in active_frame_indices]
        left_embeddings = left_embeddings[0::frame_frequency]
        right_embeddings = right_embeddings[0::frame_frequency]
        embeddings = np.concatenate((left_embeddings, right_embeddings), axis=1)

        np_stacked_array = np.stack(embeddings)
        tensor = torch.from_numpy(np_stacked_array)
        # trX = torch.stack(embeddings).float()
        return tensor, self.labels[idx] 


In [ ]:
# image_dataset = CustomImageDataset()

# train_dataset, test_dataset = torch.utils.data.random_split(image_dataset, [0.85, 0.15])

train_dataset = CustomImageDataset('/media/osero/SamsungSSD/pickles/features_left_hand_frames_small_train.pickle', '/media/osero/SamsungSSD/pickles/features_face_frames_small_train.pickle' )
test_dataset = CustomImageDataset('/media/osero/SamsungSSD/pickles/features_left_hand_frames_small_test.pickle', '/media/osero/SamsungSSD/pickles/features_face_frames_small_test.pickle'  )

cc = 5


In [ ]:
batch_size = 1
num_workers = 4

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)  # Adjust batch size as needed
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
class_names = train_dataset.classes
class_names

input_dim = train_dataset[0][0][0].size(0)  # Get input dimension from a single feature from a video
num_classes = len(set(train_dataset.classes))
print("input_dim: ", input_dim, " num_classes: ", num_classes)
print("train_dataset size: ", len(train_dataset))
print("test_dataset size: ", len(test_dataset))

## Model

In [ ]:
# class DinoVisionTransformerClassifier(nn.Module):
#     def __init__(self, input_dim, num_classes):
#         super(DinoVisionTransformerClassifier, self).__init__()
#         self.classifier = nn.Sequential(
#             nn.Linear(input_dim, 256),
#             nn.ReLU(),
#             nn.Linear(256, num_classes)
#         )
    
#     def forward(self, x):
#         x = self.classifier(x)
#         return x
    
# model = DinoVisionTransformerClassifier(input_dim=input_dim, num_classes=num_classes)
# model = model.to(device)


class VideoClassifierLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes):
        super(VideoClassifierLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, bidirectional=False, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # LSTM expects input shape: (batch, seq, features)
        _, (hidden, _) = self.lstm(x)  # Use last hidden state
        output = self.dropout(hidden[-1])
        output = self.fc(output)  # Take hidden state of the last LSTM layer
        return output
    
hidden_dim = 512
num_layers = 2
model = VideoClassifierLSTM(input_dim=input_dim, hidden_dim=hidden_dim, num_layers=num_layers, num_classes=num_classes)
model = model.to(device)

## Functions

In [ ]:
def test_images():
    correct = 0
    top_5_correct = 0
    total = 0
    running_loss = 0.0
    # since we're not training, we don't need to calculate the gradients for our outputs
    test_predicted = []
    test_labels = []

    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            labels = labels.to(device)

            # calculate outputs by running images through the network
            outputs = model(features)
            loss = criterion(outputs, labels)
            
            # the class with the highest energy is what we choose as prediction
            _, predicted = torch.topk(outputs.data, 1)
            _, predicted_top_5 = torch.topk(outputs.data, 5)
            total += labels.size(0)
            correct += (predicted.flatten()== labels.flatten()).sum().item() 
            top_5_correct += (predicted_top_5.to(device) == labels).any().sum().item()
            running_loss += loss.item()

            test_labels += (labels.cpu().numpy().tolist())
            test_predicted += (predicted.cpu().numpy().tolist())

    avg_loss = running_loss / total
    accuracy = 100 * correct / total
    top_5_accuracy = 100 * top_5_correct / total
    print(f'Accuracy of the network on the {len(test_loader)*batch_size} test video: {accuracy:.4f} %, top5: {top_5_accuracy:.4f} %, avg_loss: {avg_loss}')
    return accuracy, top_5_accuracy, avg_loss

In [ ]:
import datetime
from time import gmtime, strftime
def get_current_time():
    return strftime("%Y-%m-%d_%H-%M-%S", gmtime())

def save_model_result(current_time):
    result_name = 'lstm_results/LSTM_LF_' + current_time + '.pth'
    torch.save({'name': result_name,
                'model_state_dict': model.state_dict(),
                'lr': lr,
                'step_size': step_size,
                'gamma': gamma,
                'weight_decay': weight_decay,
                'hidden_dim': hidden_dim,
                'num_layers': num_layers,
                'batch_size': batch_size,
                'frame_frequency': frame_frequency,
                'input_dim': input_dim,
                'num_classes': num_classes,
                'train_dataset': len(train_dataset),
                'test_dataset': len(test_dataset),
                'avg_loss_list': avg_loss_list,
                'avg_accuracy_list': avg_accuracy_list,
                'avg_test_accuracy_list': avg_test_accuracy_list,
                'avg_top5_test_accuracy_list': avg_top5_test_accuracy_list,
                'avg_test_loss_list': avg_test_loss_list},
                result_name)



In [ ]:
import shutil
def copy_ipynb_file(current_time): 
    current_file = 'dino_lstm_left_face.ipynb'
    copy_file = '/home/osero/Desktop/CMPE/dinov2/classsification/lstm/lstm_results/ipynbs/COPY_' + current_time + '_' + current_file
    shutil.copy(current_file, copy_file)

## Train

In [ ]:
lr = 0.0002
step_size = 10
gamma = 0.5
weight_decay = 0

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=lr)
scheduler = lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma) ## CosineAnnealingLR Dene
print(f"lr {lr}, step_size: {step_size}, gamma: {gamma}, weight_decay: {weight_decay}")
print(f"Model hidden_dim {hidden_dim}, num_layers: {num_layers}")
print(f"batch_size {batch_size}, frame_frequency: {frame_frequency}")

avg_loss_list = []
avg_accuracy_list = []
avg_test_accuracy_list = []
avg_top5_test_accuracy_list = []
avg_test_loss_list = []

num_epoch = 30
for epoch in range(num_epoch):
    train_acc = 0
    train_loss = 0
    loop = tqdm(train_loader)

    running_loss = 0.0
    running_accuracy= 0.0
    for idx, (features, labels) in enumerate(loop):
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)

        predictions = outputs.argmax(dim=1, keepdim=True).squeeze()
        correct = (predictions == labels).sum().item()
        accuracy = correct / batch_size

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_accuracy += 100 * accuracy
        loop.set_description(f"Epoch [{epoch}/{num_epoch}]")
        loop.set_postfix(loss=loss.item(), acc=accuracy)
    scheduler.step()
    avg_loss = running_loss / len(train_loader)
    avg_accuracy = running_accuracy / len(train_loader)
    print(f"Time: {get_current_time()} Epoch [{epoch}], Avg loss: {avg_loss:.4f}, Avg accuracy: {avg_accuracy:.4f}")
    avg_test_accuracy, avg_top5_test_accuracy, avg_test_loss = test_images()

    avg_loss_list.append(avg_loss)
    avg_accuracy_list.append(avg_accuracy)
    avg_test_accuracy_list.append(avg_test_accuracy)
    avg_top5_test_accuracy_list.append(avg_top5_test_accuracy)
    avg_test_loss_list.append(avg_test_loss)
current_time = get_current_time()
save_model_result(current_time)
copy_ipynb_file(current_time)

In [ ]:
import matplotlib.pyplot as plt
import torch

# summarize history for accuracy
plt.plot(avg_accuracy_list) 
plt.plot(avg_test_accuracy_list)
plt.plot(avg_top5_test_accuracy_list)
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['Train', 'Test', 'Test Top-5'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(avg_loss_list)
plt.plot(avg_test_loss_list)
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

## Test

In [ ]:

# test_images()

## Report

In [ ]:
# print(classification_report(test_labels, test_predicted, target_names=class_names))


In [ ]:
# cm = confusion_matrix(test_labels, test_predicted)
# df_cm = pd.DataFrame(
#     cm, 
#     index = class_names,
#     columns = class_names
# )
# df_cm

In [ ]:
# def show_confusion_matrix(confusion_matrix):
#     hmap = sns.heatmap(confusion_matrix, annot=True, fmt="d", cmap="Blues")
#     plt.ylabel("Surface Ground Truth")
#     plt.xlabel("Predicted Surface")
#     plt.legend()
    
# show_confusion_matrix(df_cm)